# Dataset Validation v2 — 450 items over the real corpus

Runs all six checks on `qa_pairs_v2.json` against `corpus_v2.json`, and breaks every result down by **pilot vs. wikipedia** subset.

**Upload first:** `corpus_v2.json`, `qa_pairs_v2.json`. **Secret needed:** `GROQ_API_KEY`.

### Install

In [ ]:
# !pip install -q pandera sentence-transformers groq pandas

### Config

In [ ]:
CONFIG = {
    "alignment_similarity_threshold": 0.55,
    "near_duplicate_threshold": 0.92,
    "trivial_pair_threshold": 0.985,
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "darija_markers": [
        "ديال","بزاف","واخا","دابا","غادي","كنبغي","ماشي","شنو","علاش","فين","كيفاش",
        "بغيت","نتا","نتي","حنا","زعما","خاصني","كاين","ماكاينش","منين","بأش","واش",
        "بشحال","شحال","إمتى","شكون","فأش","فأي","فوقاش","علاه","أشمن","أش","ملي","حيت",
        "راه","بحال","هادشي","ديك","هاد","هادا","هادي","نتوما","باقي","دغيا","بلا","خلا",
        "كيدير","كتدير","كيوفر","كتوفر","كيحدد","كيساهم","كيجيو","كيزورو","كيستافد",
        "كيبدا","كيوقف","كيتقام","كيشمل","كتعاون","كتستعمل","كيتسجل","كتساقط","كيكثر",
    ],
}

### Load corpus and QA pairs (upload both files to the Colab session)

In [ ]:
import json, pandas as pd

with open("corpus_v2.json", encoding="utf-8") as f:
    corpus = json.load(f)
with open("qa_pairs_v2.json", encoding="utf-8") as f:
    qa_pairs = json.load(f)

corpus_map = {c["chunk_id"]: c["text"] for c in corpus}
KNOWN_CHUNK_IDS = set(corpus_map)

df = pd.DataFrame(qa_pairs)
df["source_chunk_text"] = df["source_chunk_id"].map(corpus_map).fillna("")
df["subset"] = df["source_chunk_id"].str.startswith("wiki_").map({True: "wikipedia", False: "pilot"})

print(f"Corpus: {len(corpus)} passages | QA items: {len(df)}")
print(df["subset"].value_counts().to_string())

### Check 1: schema and referential integrity

In [ ]:
import pandera as pa
from pandera import Column, Check, DataFrameSchema

schema = DataFrameSchema({
    "id": Column(str, unique=True, nullable=False),
    "msa_query": Column(str, Check.str_length(min_value=3), nullable=False),
    "darija_query": Column(str, Check.str_length(min_value=3), nullable=False),
    "gold_answer": Column(str, Check.str_length(min_value=1), nullable=False),
    "source_chunk_id": Column(str, nullable=False),
}, strict=False)

schema_errors = []
try:
    schema.validate(df, lazy=True)
    print("Schema check passed.")
except pa.errors.SchemaErrors as e:
    schema_errors = e.failure_cases
    print("Schema issues:")
    print(schema_errors[["column", "check", "failure_case", "index"]])

bad_refs = df[~df["source_chunk_id"].isin(KNOWN_CHUNK_IDS)]
print(f"Unresolved source_chunk_id: {len(bad_refs)}")
if len(bad_refs):
    print(bad_refs[["id", "source_chunk_id"]].to_string())

### Embedding model (used by checks 2 and 5)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

embedder = SentenceTransformer(CONFIG["embedding_model"])

msa_emb    = embedder.encode(df["msa_query"].tolist(),    normalize_embeddings=True, show_progress_bar=True, batch_size=64)
darija_emb = embedder.encode(df["darija_query"].tolist(), normalize_embeddings=True, show_progress_bar=True, batch_size=64)
print("Query embeddings computed.")

### Check 2: MSA/Darija alignment and trivial pairs

In [ ]:
sims = (msa_emb * darija_emb).sum(axis=1)
df["alignment_sim"] = sims.round(3)

def flag_alignment(s):
    if s < CONFIG["alignment_similarity_threshold"]:
        return "LOW_ALIGNMENT"
    if s > CONFIG["trivial_pair_threshold"]:
        return "TOO_SIMILAR"
    return ""

df["alignment_flag"] = df["alignment_sim"].map(flag_alignment)
flagged_align = df[df["alignment_flag"] != ""]
print(f"Alignment-flagged: {len(flagged_align)} / {len(df)}")
print(df.groupby("subset")["alignment_sim"].describe()[["mean", "min", "max"]].to_string())
print(flagged_align.groupby(["subset", "alignment_flag"]).size().to_string())

### Check 3: dialect authenticity (lexicon scan)

In [ ]:
def marker_hits(text):
    return sum(1 for m in CONFIG["darija_markers"] if m in text)

df["darija_marker_hits"] = df["darija_query"].map(marker_hits)
weak_dialect = df[df["darija_marker_hits"] == 0]
print(f"Zero Darija markers: {len(weak_dialect)} / {len(df)}")
print(df.groupby("subset")["darija_marker_hits"].mean().round(2).to_string())
if len(weak_dialect):
    print(weak_dialect[["id", "subset", "darija_query"]].head(25).to_string())

### Check 4: answer grounding via Groq LLM-as-judge

In [ ]:
import time
from groq import Groq, RateLimitError
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

def judge(answer, source_text, model="openai/gpt-oss-20b", max_retries=5):
    prompt = (
        f'Source text: "{source_text}"\n'
        f'Claim: "{answer}"\n\n'
        "Is the claim fully and accurately supported by the source text? "
        "Check dates, numbers and names carefully.\n"
        "Think briefly, then answer with exactly one word: YES or NO."
    )
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=model, messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=400, reasoning_effort="low",
            )
            return "YES" if "YES" in (r.choices[0].message.content or "").upper() else "NO"
        except RateLimitError:
            wait = 20 * (attempt + 1)
            print(f"  Rate limited, waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"  Judge failed ({e}); marking UNKNOWN.")
            return "UNKNOWN"
    return "UNKNOWN"

verdicts = []
for i, row in df.iterrows():
    verdicts.append(judge(row["gold_answer"], row["source_chunk_text"]))
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(df)}")

df["grounded"] = verdicts
ungrounded = df[df["grounded"] == "NO"]
print(f"\nUngrounded: {len(ungrounded)} / {len(df)}")
print(df.groupby(["subset", "grounded"]).size().to_string())

### Check 5: near-duplicate detection

In [ ]:
sim_matrix = msa_emb @ msa_emb.T
np.fill_diagonal(sim_matrix, 0)
pairs = np.argwhere(sim_matrix > CONFIG["near_duplicate_threshold"])
seen, dup_rows = set(), []
for i, j in pairs:
    key = tuple(sorted((i, j)))
    if key in seen:
        continue
    seen.add(key)
    dup_rows.append({
        "id_1": df.iloc[i]["id"], "id_2": df.iloc[j]["id"],
        "similarity": round(float(sim_matrix[i, j]), 3),
    })

dup_df = pd.DataFrame(dup_rows)
print(f"Near-duplicate pairs: {len(dup_df)}")
if len(dup_df):
    print(dup_df.head(30).to_string(index=False))

### Check 6: corpus coverage

In [ ]:
cov = df["source_chunk_id"].value_counts()
print(f"Distinct passages used: {len(cov)} of {len(corpus)}")
print(f"Max items on one passage: {cov.max()} ({cov.idxmax()})")
print(cov.value_counts().sort_index().rename("passages").to_string())

### Consolidated review list + export

In [ ]:
review = set()
if len(schema_errors):
    review |= set(df.iloc[schema_errors["index"].dropna().astype(int).unique()]["id"])
review |= set(bad_refs["id"])
review |= set(flagged_align["id"])
review |= set(weak_dialect["id"])
review |= set(ungrounded["id"])
for r in dup_rows:
    review |= {r["id_1"], r["id_2"]}

df["needs_review"] = df["id"].isin(review)

print("=" * 60)
print(f"VALIDATION SUMMARY — {len(df)} items")
print("=" * 60)
print(f"Flagged for review: {len(review)} ({len(review)/len(df)*100:.1f}%)")
print(df.groupby("subset")["needs_review"].agg(["sum", "count"]).to_string())

flagged = df[df["needs_review"]][
    ["id", "subset", "msa_query", "darija_query", "gold_answer",
     "source_chunk_id", "alignment_sim", "alignment_flag",
     "darija_marker_hits", "grounded"]
]
flagged.to_json("flagged_v2.json", orient="records", force_ascii=False, indent=2)
df.to_csv("validation_full_v2.csv", index=False)
print("\nWrote flagged_v2.json and validation_full_v2.csv")

from google.colab import files
files.download("flagged_v2.json")
files.download("validation_full_v2.csv")